# 选题二：语言模型先验（Colab + 免费 T4）

**这个 notebook 只做一件事**：在 Colab 的 T4 上，用 vLLM 批量生成语言模型先验，
并把缓存与结果通过 git 回传仓库。其余分析（评测、融合实验、画图、出文档）都在本机做。

**为什么用 git 而不是上传压缩包**：仓库每次改动只需要 `git pull` 一行，
不会出现"压缩包版本和本地不一致""路径分隔符在 Linux 下失效"这类问题；
结果与缓存也随提交回到仓库，导师和同组同学能直接看到。

**一次性准备**

1. 把仓库推到 GitHub 或 Gitee（本机执行一次）：
   `git remote add origin <仓库地址>` + `git push -u origin <分支>`；
2. 在 Colab 左侧密钥面板里配两个 Secret（没有面板就跳过 HF 那个，不影响 Qwen）：
   * `HF_TOKEN`：Hugging Face read token，避免下载限速；
   * `GIT_TOKEN`：仓库的写权限 token（GitHub 用 Personal Access Token，
     Gitee 用私人令牌），用于把结果推回去；
3. 如果仓库是私有的，克隆时把 `GIT_TOKEN` 拼进 URL（下一步的代码已经这么写）。

**每个会话的流程**：配置参数 → 拉仓库 → 一键搭环境 → 探测模型 → 生成 → 回传 → 校验。
整个环境（含 vLLM）在 Colab 上是临时的，所以每个会话都要重跑第 5 格；
但**生成结果是断点续跑的**，缓存已在仓库里，重复的部分会自动跳过。

## 1. 参数配置（只改这一格）

In [ ]:
# 仓库地址（已按你的仓库填好，一般不用改）
REPO_URL = 'https://github.com/yyyyyyy171/causal-llm.git'
BRANCH   = 'yyyyyyy171/topic2'

# 本次要跑的网络：先用最小的 ASIA 验证通路，再换 CHILD / INSURANCE
NETWORK  = 'ASIA'

# 先跑 ASIA 验证通路，再换 CHILD / INSURANCE
REPO = '/content/Causal-LLM'
VENV = '/content/venv'

## 2. 确认显卡与驱动

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!python -c "import sys; print('python', sys.version.split()[0])"

## 3. 拉取或更新仓库

第一次是 `clone`，之后每个会话也是 `clone`（虚拟机是新的）；
要是想在同一会话里同步最新改动，把这格改成 `git pull` 那句即可。

In [ ]:
import os, subprocess

# 防呆：占位符没替换的话，GitHub/Gitee 只会回一个 400，看不出真正原因
assert '<' not in REPO_URL and REPO_URL.endswith('.git'), (
    '请先把 REPO_URL 换成你自己的仓库地址，例如 '
    'https://github.com/yourname/causal-llm.git ；当前是 ' + REPO_URL
)

try:
    from google.colab import userdata
    token = userdata.get('GIT_TOKEN') or ''
except Exception:
    token = ''

# 从网页复制 token 常会带上换行/空格，不清掉的话 URL 就是坏的
# （之前就是这样：https://ghp_xxx\\r\\n@github.com/... 直接 clone 失败）
token = ''.join(token.split())

# 凭据只写进虚拟机的 ~/.git-credentials，由 git 自己读取：
#   * 不写进 remote URL —— 失败时 git 不会把 token 打进错误信息；
#   * 不放进命令行参数 —— 不会出现在进程列表和日志里；
#   * 该文件在 Colab 临时虚拟机里，不会被提交进仓库。
if token:
    with open(os.path.expanduser('~/.git-credentials'), 'w', encoding='utf-8') as handle:
        handle.write(f'https://{token}@github.com\n')
    subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
    print(f'已配置 git 凭据（token 长度 {len(token)}，不打印内容）')
else:
    print('没有配置 GIT_TOKEN：可以克隆公开仓库，但无法把结果推回仓库')

if os.path.exists(REPO):
    # Colab 里通常会先产生本地提交（生成的缓存/结果），历史与远端分叉，
    # --ff-only 会直接失败且什么都不做。这里用 rebase 合并，并且**把结果打印出来**：
    # 之前用 check=False 静默吞掉了失败，导致"以为拉到最新、其实脚本还是旧的"。
    print('仓库已存在，拉取最新改动 ...')
    result = subprocess.run(
        ['git', '-C', REPO, 'pull', '--rebase', '--autostash'],
        capture_output=True, text=True)
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        print('!! 拉取失败：上面的输出说明了原因，处理完再继续，否则跑的是旧代码')
else:
    # 克隆用干净的地址；公开仓库不需要 token，私有仓库由上面的凭据自动补上
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, REPO], check=True)

# 提交身份（Colab 虚拟机里需要现配）
subprocess.run(['git', '-C', REPO, 'config', 'user.email', 'colab@local'])
subprocess.run(['git', '-C', REPO, 'config', 'user.name', 'colab'])

# 自证代码是最新的：脚本里应该含这两处修复，否则后面白跑
script_path = os.path.join(REPO, 'scripts', 'run_llm_prior_vllm.py')
script_text = open(script_path, encoding='utf-8').read()
print('脚本版本自检: chat_template_kwargs =', 'chat_template_kwargs' in script_text,
      '| finish_reason =', 'finish_reason' in script_text)

print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-3'],
                     capture_output=True, text=True).stdout)
print('已有缓存:')
for network in sorted(os.listdir(os.path.join(REPO, 'data'))):
    cache_dir = os.path.join(REPO, 'data', network, 'llm_cache')
    if os.path.isdir(cache_dir):
        for name in sorted(os.listdir(cache_dir)):
            with open(os.path.join(cache_dir, name), encoding='utf-8') as handle:
                print(f'  {network}/{name}: {sum(1 for _ in handle)} 条')

## 4. 一键搭建环境

环境由 `scripts/setup_colab.sh` 负责：建隔离虚拟环境、固定 vLLM 版本，
并在驱动低于 580 时自动把 torch 换成 CUDA 12.1 版本
（CUDA 13 的 torch 需要驱动 ≥580，Colab 的 T4 常常达不到，
这正是 vLLM 报 `Engine core initialization failed` 的常见原因）。

In [ ]:
!cd /content/Causal-LLM && bash scripts/setup_colab.sh

## 5. 登录 Hugging Face（可选）

`Qwen/Qwen3-8B-AWQ` 是公开仓库，匿名就能下载；配上 `HF_TOKEN` 只是避免限速。

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    from huggingface_hub import login, whoami
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('已登录:', whoami().get('name'))
except Exception as error:
    print('未配置 HF_TOKEN，匿名下载（Qwen 系列可以忽略）:', type(error).__name__)

## 6. 选模型

优先 8B 的 4bit 量化版——文档要求 8B，而 T4 的 16GB 放不下纯 fp16 的 8B
（约 16GB 权重），放 AWQ 版（约 5.7GB）很从容。

In [ ]:
from huggingface_hub import model_info

CANDIDATES = [
    'Qwen/Qwen3-8B-AWQ',           # 8B 4bit，已确认存在（下载量 200 万+）
    'Qwen/Qwen3-4B-Instruct-2507', # 4B fp16，兜底（约 8GB）
]

MODEL = None
for candidate in CANDIDATES:
    try:
        model_info(candidate)
        MODEL = candidate
        print('可用:', candidate)
        break
    except Exception as error:
        print('不可用:', candidate, f'({type(error).__name__})')

print()
print('本会话将使用:', MODEL)

## 7. 生成先验

用 venv 的解释器运行，避免与 Colab 预装包冲突。

In [ ]:
import subprocess

assert MODEL, '上一格没探测到可用模型'

command = [
    f'{VENV}/bin/python', 'scripts/run_llm_prior_vllm.py',
    '--network', NETWORK,
    '--mode', 'both',          # 真实名 + 匿名，匿名的用于语义消融
    '--task', 'direction',     # 想换成"是否同属马尔可夫毯"就改成 mb
    '--model', MODEL,
    '--enforce-eager',         # T4（Turing）上关掉 CUDA 图捕获，避免引擎初始化失败
    '--gpu-memory-utilization', '0.85',
    '--max-model-len', '1024',
]

print('运行:', ' '.join(command))
print()
subprocess.run(command, check=True, cwd=REPO)

## 8. 把结果提交回仓库

缓存与聚合结果都在 `data/<网络>/` 下，属于可复现证据，随 git 回传。
需要 `GIT_TOKEN`（写权限）。token 存在 `~/.git-credentials` 里、由 git 读取，
**不进命令行也不进 remote**，所以这里失败的报错里不会带 token。
没配 token 时这一格只在本地提交，不推送。

In [ ]:
import subprocess

message = f'Colab 生成 {NETWORK} 先验（{MODEL}）'

subprocess.run(['git', '-C', REPO, 'add', '-A'], check=True)
commit = subprocess.run(['git', '-C', REPO, 'commit', '-m', message],
                        capture_output=True, text=True)
print(commit.stdout.strip() or commit.stderr.strip())

if not token:
    print('没有 GIT_TOKEN，跳过推送。结果已在虚拟机的仓库里，可从左侧文件面板下载。')
else:
    # 注意：这里绝对不能把 token 拼进命令或 URL，否则一旦失败，
    # token 会被 git 打进错误信息（我们就是这样泄露过一次）。
    push = subprocess.run(['git', '-C', REPO, 'push', 'origin', BRANCH],
                          capture_output=True, text=True)
    print(push.stdout.strip() or push.stderr.strip())

## 9. 校验结构化输出

In [ ]:
!cd /content/Causal-LLM && python scripts/validate_json_output.py